# Limpieza inicial — antecedente

Esta es la copia inicial del trabajo de limpieza. Conserva decisiones preliminares y rutas de Colab del momento en que se hizo. Para reproducir el proyecto se usa [el notebook principal](../Proyecto_Final_B_Portafolio_Cine.ipynb), que trabaja con los CSV del repositorio. No combinar resultados de este archivo con la propuesta final.


### Mount Google Drive

Run the cell below to mount your Google Drive. You will be prompted to authorize access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Once mounted, you can list or load files using their path starting with `/content/drive/MyDrive/`.

In [ ]:
from pathlib import Path
import pandas as pd

# Ubicación corregida de los archivos utilizando espacios en lugar de guiones bajos
RUTA_DATOS = Path("/content/drive/MyDrive/Colab Notebooks/Programacion analitica 2/Proyecto final/cvs")

# Tipos esperados para las columnas de películas
tipos_peliculas = {
    "id_pelicula": "Int64",
    "titulo": "string",
    "anio_estreno": "Int64",
    "fecha_estreno": "string",
    "genero": "string",
    "clasificacion": "string",
    "pais_origen": "string",
    "es_secuela": "string",
    "director": "string",
    "guionista": "string",
    "protagonista": "string",
    "productora": "string",
    "duracion_minutos": "Float64",
    "puntaje_imdb": "Float64",
    "cantidad_votos": "Float64",
    "presupuesto_usd": "Float64",
    "recaudacion_mundial_usd": "Float64",
}

# Cargar los datasets
peliculas = pd.read_csv(
    RUTA_DATOS / "peliculas.csv",
    dtype=tipos_peliculas
)

ipc = pd.read_csv(
    RUTA_DATOS / "ipc_eeuu.csv",
    dtype={"anio": "Int64", "ipc_eeuu": "Float64"}
)

In [ ]:
# Observar una muestra aleatoria de 30 fechas
peliculas["fecha_estreno"].sample(30, random_state=42)

Las fechas estan escritas con dos formatos distintos:
- 24/05/2002
- 1988-07-01


In [ ]:

print("DIMENSIONES")
print("Películas:", peliculas.shape)
print("IPC:", ipc.shape)

print("\nPRIMERAS FILAS DE PELÍCULAS")
display(peliculas.head())

print("\nPRIMERAS FILAS DEL IPC")
display(ipc.head())

print("\nTIPOS DE DATOS - PELÍCULAS")
peliculas.info()

print("\nTIPOS DE DATOS - IPC")
ipc.info()


In [ ]:
peliculas["fecha_estreno"] = pd.to_datetime(
    peliculas["fecha_estreno"],
    format="mixed",
    dayfirst=True
)

In [ ]:
# revision despues de adjustar las fechas.
peliculas["fecha_estreno"].sample(30, random_state=42)

Formatos diferentes en fechas de estreno

Problema detectado: la misma columna contiene fechas en formatos YYYY-MM-DD y DD/MM/YYYY.

Decision: interpretar ambos formatos explicitamente y unificarlos en datetime64[ns].

Justificacion: permitir un analisis temporal consistente sin alterar las fechas originales.

Registros eliminados: 0.

In [ ]:
peliculas.describe()

#algunas observaciones sospechosas

- Duracion_minutos: No puede haber una pelicula que dure 0 minutos. Por lo tanto hay que investigar que esta ocurriendo con el valor minimo.

- Puntaje_imdb: Lo mismo que duracion_minutos, el valor minimo posible de puntaje es de 1, por lo tanto hay que observar que ocurre que el valor minimo registrado es 0.

- Presupuesto_usd:
  - Un presupuesto no puede ser negativo.
  - Hay valores faltantes. count = 5551 de 7740 registros (faltan alrededor del 28,3% de los registros)

- Recaudacion_mundial_usd: una recaudacion no puede ser negativa.

Adicionalmente el promedio de recaudacion es considerablemente mayor a la mediana. Esto podria indicar que unas pocas peliculas muy exitosas suben el promedio.

In [ ]:
#investiguemos los valores negativos de presupuesto
peliculas[peliculas["presupuesto_usd"] < 0]

Obsrvamos que hay 22 peliculas que tienen un presupuesto de -1. Estas representan aproximadamente el 0,28% del dataset.

Es muy probable que -1 represente un presupuesto desconocido, en lugar de un presupuesto real.

Decision: reemplazar esos -1 por NaN
- Razonamiento:
  - Un presupuesto negativo es imposible.
  - No conocemos el presupuesto verdadero.
  - No queremos inventar datos ni eliminar peliculas innecesariamente.

In [ ]:
#Investigemos los valores negativos de la recaudacion

peliculas[peliculas["recaudacion_mundial_usd"] < 0]

Observamos 11 peliculas con recaudacion negativa (aprox. 0,14% del dataset)

A diferencia de los presupuestos negativos, donde siempre aparecia -1, aca encontramos distintos valores. Esto puede sugerir un error en signo. Corroborando la recaudacion de Ant man and the Wasp con IMBD PRO, podemos presenciar que efectivamente los $622674139.0 vienen siendo la recauadcion positiva.


Tenemos dos alternativas:
1. Aplicar abs() para convertir los numeros a positivos, si y solo si, confirmamos que el valor es correcto y el signo esta equivocado para todas las peliculas.
2. Convertirlos a NaN si no podemos determinar su recaudacion verdadera.

In [ ]:
#investigemos filas duplicadas. Indirectamente podemos verificar si las peliculas con recauadcion negativa no tienen otra entrada duplicada con el valor de recaudacion en positivo.
peliculas[
    peliculas["id_pelicula"].duplicated(keep=False)
].sort_values("id_pelicula")

In [ ]:
# Cantidad de duplicados exactos
print("Duplicados exactos:", peliculas.duplicated().sum())

# Cantidad de filas adicionales con ID repetido
print(
    "IDs repetidos:",
    peliculas.duplicated(subset="id_pelicula").sum()
)

#Si ambos resultados coinciden, significaria que todas las repeticiones por ID corresponden a duplicados exactos.

#Si son diferentes, existen peliculas repetidas con datos distintos.

Al ser duplicados exactos, lo mas recomendable es quitarlas. Vamos a hacer esto en una copia del dataset.

In [ ]:
# Guardamos la cantidad original
filas_antes = len(peliculas)

# Eliminamos duplicados en una copia
peliculas_limpias = peliculas.drop_duplicates().copy()

# Medimos el impacto
filas_eliminadas = filas_antes - len(peliculas_limpias)
porcentaje = filas_eliminadas / filas_antes * 100

print("Filas eliminadas:", filas_eliminadas)
print(f"Porcentaje eliminado: {porcentaje:.2f}%")
print("Filas restantes:", len(peliculas_limpias))

In [ ]:
#investigamos duraciones sospechosas
duraciones_sospechosas = peliculas_limpias[
    (peliculas_limpias["duracion_minutos"] <= 0) |
    (peliculas_limpias["duracion_minutos"] > 300)
]

display(
    duraciones_sospechosas[
        ["titulo", "anio_estreno", "duracion_minutos"]
    ].sort_values("duracion_minutos")
)

Observamos dos tipos de valores imposibles.
- peliculas con una duracion de 0 minutos
- peliculas con una duracion de 999 minutos

Posible resolucion:

Convertir los valores 0 y 999 a NaN, porque desconocemos las duraciones verdaderas.

Conservamos por ahora las de 357 y 366 minutos ya que presentan duraciones posibles.

In [ ]:
# Identificar duraciones imposibles
duraciones_invalidas = peliculas_limpias["duracion_minutos"].isin([0, 999])

# medimos cuanto impactan
cantidad = duraciones_invalidas.sum()
porcentaje = cantidad / len(peliculas_limpias) * 100

print(f"Registros afectados: {cantidad}")
print(f"Porcentaje afectado: {porcentaje:.2f}%")

# Reemplazar valores invalidos por faltantes
peliculas_limpias.loc[duraciones_invalidas, "duracion_minutos"] = pd.NA

In [ ]:
#investigamos las peliculas con rating de IMBD por afuera de la escala (1-10)
puntajes_invalidos = peliculas_limpias[
    (peliculas_limpias["puntaje_imdb"] < 1) |
    (peliculas_limpias["puntaje_imdb"] > 10)
]

display(
    puntajes_invalidos[
        ["titulo", "anio_estreno", "puntaje_imdb", "cantidad_votos"]
    ]
)

print("Cantidad de registros:", len(puntajes_invalidos))

Observamos:
- 22 películas tienen un puntaje IMDb de 0.
- Todas tienen votos registrados.
- Representan aproximadamente el 0,29% de las 7.682 filas.

Lo mas probable es que el valor de 0 halla sido utilisado para representar puntuajes desconocidos, por lo tanto:

Convertirmos los ceros a NaN, ya que desconocemos las calificaciones verdaderas.

In [ ]:
puntajes_invalidos = peliculas_limpias["puntaje_imdb"] == 0

# Medir impacto
cantidad = puntajes_invalidos.sum()
porcentaje = cantidad / len(peliculas_limpias) * 100

print(f"Registros afectados: {cantidad}")
print(f"Porcentaje: {porcentaje:.2f}%")

# Reemplazar los ceros por valores faltantes
peliculas_limpias.loc[puntajes_invalidos, "puntaje_imdb"] = float("nan")

In [ ]:
#Anteriormente encontramos 22 peliculas con presupuesto de -1. Como un presupuesto negativo es imposible y no conocemos el valor verdadero, vamos a reemplazar esos -1 por valores nan.

# Identificar presupuestos negativos
presupuestos_invalidos = peliculas_limpias["presupuesto_usd"] < 0

# Medir impacto antes de corregir
faltantes_antes = peliculas_limpias["presupuesto_usd"].isna().sum()
cantidad = presupuestos_invalidos.sum()

# Reemplazar por NaN
peliculas_limpias.loc[presupuestos_invalidos, "presupuesto_usd"] = float("nan")

# Comprobar el resultado
faltantes_despues = peliculas_limpias["presupuesto_usd"].isna().sum()

print("Presupuestos corregidos:", cantidad)
print("Faltantes antes:", faltantes_antes)
print("Faltantes despues:", faltantes_despues)
print(f"Porcentaje afectado: {cantidad / len(peliculas_limpias) * 100:.2f}%")

Habiamos previamente identificado 22 presupuestos negativos (-1), interpretados como posibles indicadores de datos desconocidos. Se decidio reemplazarlos por NaN para evitar distorsionar los futuros calculos de rentabilidad.

In [ ]:
#Habiamos encontrado 11 peliculas con recaudaciones negativas. A diferencia de los presupuestos, estos valores no son todos -1.

#Podria ser un errores de signo, pero tenemos que verificarlo.

#Primero vamos a comprobar si esas mismas peliculas tienen otra recaudacion registrada.

recaudaciones_negativas = peliculas_limpias[
    peliculas_limpias["recaudacion_mundial_usd"] < 0
]

# Buscar otras filas con los mismos titulos
peliculas_limpias[
    peliculas_limpias["titulo"].isin(
        recaudaciones_negativas["titulo"]
    )
][
    ["titulo", "anio_estreno", "recaudacion_mundial_usd"]
].sort_values("titulo")

Esta investigacion nos muestra que no hay mas de una recaudacion registrada para estas peliculas que continenen una recaudacion negativa. Pero como investigamos previamente con "Antman and the Wasp" descubrimos que el valor de recaudacion era el mismo que el que presenta Box Office Mojo pero con cambio de signo, asi que combiene hacer una investigacion para cada una de las 7 peliculas.

In [ ]:
# Corregir el signo de las recaudaciones verificadas
recaudaciones_negativas = peliculas_limpias["recaudacion_mundial_usd"] < 0

peliculas_limpias.loc[
    recaudaciones_negativas, "recaudacion_mundial_usd"
] = peliculas_limpias.loc[
    recaudaciones_negativas, "recaudacion_mundial_usd"
].abs()

print("Registros corregidos:", recaudaciones_negativas.sum())
print("Negativos restantes:",
      (peliculas_limpias["recaudacion_mundial_usd"] < 0).sum())

Se detectaron 11 peliculas con recaudaciones negativas. Para investigar si se trataba de errores de signo, comparamos los valores con fuentes externas como Box Office Mojo.

Encontramos que la mayoria (9) coincidian exactamente con las cifras publicadas, pero con signo negativo. En dos casos, los importes coincidian con la recaudación domestica y no con la mundial.

Con esta evidencia, corregimos los signos (de las 9) y dejamos documentadas las diferencias encontradas.

In [ ]:
peliculas_limpias.loc[
    peliculas_limpias["titulo"] == "Planes, Trains & Automobiles",
    "recaudacion_mundial_usd"
] = 49530552

peliculas_limpias.loc[
    peliculas_limpias["titulo"] == "The Long Day Closes",
    "recaudacion_mundial_usd"
] = 22885

Para las dos peliculas cuyas recaudaciones correspondian al mercado domestico  reemplazamos el valor por sus recaudaciones mundiales verificadas en fuentes externas, manteniendo la consistencia de la variable.

In [ ]:
#investigacion de las variables categoricas

columnas_categoricas = [
    "genero",
    "clasificacion",
    "pais_origen",
    "es_secuela"
]

for columna in columnas_categoricas:
    print(f"\n--- {columna.upper()} ---")
    print(peliculas_limpias[columna].value_counts(dropna=False))

Inconsistencias detectadas:
- Comedia y comedia representan lo mismo.
- Acción y ACCION representan lo mismo.
- Biografía y Biografia representan lo mismo.
- ATP y Apta todo publico representan lo mismo.
- Estados Unidos y EE.UU. representan lo mismo.

In [ ]:
#estandarizacion de los generos
correcciones_genero = {
    "comedia": "Comedia",
    "ACCION": "Acción",
    "POLICIAL": "Policial",
    "Biografia": "Biografía",
    "Animacion": "Animación",
    "Ciencia Ficcion": "Ciencia ficción"
}

peliculas_limpias["genero"] = peliculas_limpias["genero"].replace(
    correcciones_genero
)

# Comprobar el resultado
print(peliculas_limpias["genero"].value_counts())

Terror se encuentra dos veces. Suponemos se debe a espacios invicibles.

In [ ]:
#eliminamos los posibles espacios invicibles con .str.strip()
peliculas_limpias["genero"] = peliculas_limpias["genero"].str.strip()

print(peliculas_limpias["genero"].value_counts())

In [ ]:
#normalizacion de las clasificaciones

correcciones_clasificacion = {
    "MAYORES DE 17": "Mayores de 17",
    "Supervision parental": "Supervisión parental",
    "Apta todo publico": "ATP"
}

# Eliminar espacios innecesarios
peliculas_limpias["clasificacion"] = (
    peliculas_limpias["clasificacion"].str.strip()
)

# Unificar categorias
peliculas_limpias["clasificacion"] = (
    peliculas_limpias["clasificacion"].replace(
        correcciones_clasificacion
    )
)

# Verificar resultados
print(peliculas_limpias["clasificacion"].value_counts(dropna=False))

In [ ]:
#revisamos los valores de la columna pais_origen
print(
    peliculas_limpias["pais_origen"]
    .value_counts(dropna=False)
    .to_string()
)

Observaciones:
- 3 valores faltantes (NA) en pais_origen.

- Tambien encontramos varias inconsistencias:
  - Valores actuales
  - EE.UU. / Estados Unidos
  - UK / Reino Unido
  - Canada / Canadá
  - Japon / Japón
  - Espania / España
  - Mexico / México
  - Romania / Rumanía
  - Libya / Libia


  Empezando por las inconcistencias, vamos a estandarizarlas elejiendo un mismo nombre por pais.

In [ ]:

peliculas_limpias["pais_origen"] = (
    peliculas_limpias["pais_origen"].str.strip()
)

correcciones_pais = {
    "EE.UU.": "Estados Unidos",
    "UK": "Reino Unido",
    "Canada": "Canadá",
    "Japon": "Japón",
    "Espania": "España",
    "Mexico": "México",
    "Romania": "Rumanía",
    "Libya": "Libia"
}

peliculas_limpias["pais_origen"] = (
    peliculas_limpias["pais_origen"].replace(correcciones_pais)
)

print(peliculas_limpias["pais_origen"].value_counts(dropna=False).to_string())

In [ ]:
#investigar los países faltantes (na)
peliculas_limpias[
    peliculas_limpias["pais_origen"].isna()
][
    ["titulo", "anio_estreno", "director", "productora", "pais_origen"]
]

In [ ]:
peliculas_limpias.loc[
    peliculas_limpias["titulo"].isin([
        "Clinton Road",
        "Saw: The Final Chapter",
        "The Wolfman"
    ]),
    "pais_origen"
] = "Estados Unidos"

# Verificar
print("Países faltantes:",
      peliculas_limpias["pais_origen"].isna().sum())

Para las 3 peliculas sin pais de origen registrado, se consultaron fuentes externas y se completaron los valores. todas menos "SAW: The Final Chapter" pertenecian solamente a Estados Unidos. Saw comparte produccion con Canada. Ultimamente se dejo Estados Unidos, utilizando un criterio de pais principal para las coproducciones. Se recuperaron los tres datos faltantes sin eliminar registros.

In [ ]:
#revision de datos faltantes restantes

# Cantidad de valores faltantes por columna
faltantes = peliculas_limpias.isna().sum()

# Porcentaje de valores faltantes
porcentaje_faltantes = (
    faltantes / len(peliculas_limpias) * 100
)

# Crear tabla resumen
resumen_faltantes = pd.DataFrame({
    "Cantidad": faltantes,
    "Porcentaje": porcentaje_faltantes
})

# Mostrar columnas con faltantes, de mayor a menor
display(
    resumen_faltantes[
        resumen_faltantes["Cantidad"] > 0
    ].sort_values("Porcentaje", ascending=False)
)

Observamos:
- principal problema es presupuesto_usd, con un 28,55% de valores faltantes.

Pasos siguientes:
Identificar si son peliculas al azar o comparten caracteristicas.

In [ ]:
#investigamos los faltantes por decada, para ver si hay algun patron
# Crear una columna con la decada de estreno
peliculas_limpias["decada"] = (
    peliculas_limpias["anio_estreno"] // 10 * 10
)

# Calcular el porcentaje de presupuestos faltantes por decada
faltantes_por_decada = (
    peliculas_limpias.groupby("decada")["presupuesto_usd"]
    .apply(lambda columna: columna.isna().mean() * 100)
)

print(faltantes_por_decada)

Se presenta una situacion delicada:

los 80s tienen una proporcion considerablemente mayor de presupuestos faltantes que los años 2000.

Por lo tanto, eliminar todas las peliculas sin presupuesto podria hacer que algunas décadas esten subrepresentadas en nuestro analisis financiero.

El 40% de 2020 tambien llama la atencion.

Hay que ser cuidadosos con el procedimiento

In [ ]:
#Cantidad de peliculas con presupuesta faltante por decada

resumen_decadas = peliculas_limpias.groupby("decada").agg(
    total_peliculas=("id_pelicula", "count"),
    presupuestos_faltantes=("presupuesto_usd", lambda x: x.isna().sum()),
    porcentaje_faltante=("presupuesto_usd", lambda x: x.isna().mean() * 100)
)

display(resumen_decadas.round(2))

Los presupuestos faltantes presentan diferencias importantes segun la decada.  los 1980s tiene la mayor proporcion de faltantes entre las decadas completas, mientras que la de 2000 presenta la menor. Eliminar todas las peliculas sin presupuesto podria introducir un sesgo en el analisis de la rentabilidad.

In [ ]:
#Revisamos las faltantes por genero

resumen_generos = peliculas_limpias.groupby("genero").agg(
    total_peliculas=("id_pelicula", "count"),
    presupuestos_faltantes=("presupuesto_usd", lambda x: x.isna().sum()),
    porcentaje_faltante=("presupuesto_usd", lambda x: x.isna().mean() * 100)
)

display(
    resumen_generos.sort_values(
        "porcentaje_faltante", ascending=False
    ).round(2)
)

Los presupuestos faltantes varian bastante segun el genero.

Los hallazgos mas importantes son:
- Drama: 42,20% de faltantes (641 peliculas).
- Comedia: 33,17% (746 peliculas).
- Accion: 16,57% (283 peliculas).
- Terror: 18,01% (58 peliculas).

Consideraciones:
Si eliminaramos todas las peliculas sin presupuesto, perderiamos proporcionalmente muchas mas peliculas de Drama que de Accion.

Esto podria sesgar nuestras comparaciones de rentabilidad entre generos.

In [ ]:
#Revisamos las faltantes por pais

resumen_paises = peliculas_limpias.groupby("pais_origen").agg(
    total_peliculas=("id_pelicula", "count"),
    presupuestos_faltantes=("presupuesto_usd", lambda x: x.isna().sum()),
    porcentaje_faltante=("presupuesto_usd", lambda x: x.isna().mean() * 100)
)

display(
    resumen_paises[
        resumen_paises["total_peliculas"] >= 20
    ].sort_values(
        "porcentaje_faltante", ascending=False
    ).round(2)
)

## Observaciones importantes:
India	posee un 85,48% de faltantes

Francia posee un 62,37% de faltantes

Reino Unido	posee un 39,58% de faltantes

Estados Unidos posee un	20,33% de faltantes

si eliminamos todas las peliculas sin presupuesto, nuestro analisis quedara mas concentrado en producciones estadounidenses.

#**Conclusione final del analisis de las faltantes en presupuestos:**
Los presupuestos faltantes no se distribuyen uniformemente entre decadas, generos y paises. Por lo tanto, eliminar todos los registros incompletos podria introducir sesgos en el analisis de rentabilidad, especialmente en peliculas antiguas, dramas y producciones internacionales.

####**Posibles pasos a seguir:**

- Conservar las peliculas en el dataset general.



- No inventar presupuestos utilizando promedios o medianas.



- Excluir unicamente las peliculas sin presupuesto de los calculos que
necesariamente lo requieren, como el retorno sobre la inversion.



- Explicar que nuestras conclusiones financieras se basan en peliculas con información suficiente y podrian no representar toda la industria.